In [1]:
import pandas as pd
from docx import Document
from docx.shared import Pt, RGBColor, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
import os

BASE_DIR = r"C:\Users\axdrx\OneDrive\Escritorio\TFM\Extraccion"
estudios = pd.read_csv(os.path.join(BASE_DIR, "tabla_estudios.csv"), encoding='utf-8-sig')

# Helpers

def set_cell_bg(cell, color_hex):
    tc   = cell._tc
    tcPr = tc.get_or_add_tcPr()
    shd  = OxmlElement('w:shd')
    shd.set(qn('w:val'),   'clear')
    shd.set(qn('w:color'), 'auto')
    shd.set(qn('w:fill'),  color_hex)
    tcPr.append(shd)

def format_col_header(cell, text):
    set_cell_bg(cell, "F2F2F2")
    cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
    p = cell.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.space_before = Pt(4)
    p.paragraph_format.space_after  = Pt(4)
    run = p.add_run(text)
    run.bold = True
    run.font.size = Pt(10)
    run.font.name = "Calibri"

def format_data_cell(cell, text, alt=False):
    if alt:
        set_cell_bg(cell, "F2F2F2")
    cell.vertical_alignment = WD_ALIGN_VERTICAL.TOP
    p = cell.paragraphs[0]
    p.paragraph_format.space_before = Pt(4)
    p.paragraph_format.space_after  = Pt(4)
    run = p.add_run(str(text) if pd.notna(text) else "")
    run.font.size = Pt(10)
    run.font.name = "Calibri"
    
def cant_split_row(row):
    """Evita que una fila se parta entre páginas."""
    trPr = row._tr.get_or_add_trPr()
    cs = OxmlElement('w:cantSplit')
    cs.set(qn('w:val'), '1')
    trPr.append(cs)
    
# Crear documento

doc = Document()

for section in doc.sections:
    section.page_width    = Cm(29.7)
    section.page_height   = Cm(21.0)
    section.left_margin   = Cm(1.5)
    section.right_margin  = Cm(1.5)
    section.top_margin    = Cm(2.0)
    section.bottom_margin = Cm(2.0)
    section.orientation   = 1

h = doc.add_heading("Tabla 1. Características generales de los estudios incluidos", level=1)
h.runs[0].font.color.rgb = RGBColor(0x00, 0x00, 0x00)
h.runs[0].font.size = Pt(12)
doc.add_paragraph()

NCOLS = 4
table = doc.add_table(rows=0, cols=NCOLS)
table.style = 'Table Grid'

# Cabecera
hrow = table.add_row()
headers = ["Artículo", "Diseño del estudio",
           "Subgrupos de pacientes", "Técnicas utilizadas"]
for i, h_text in enumerate(headers):
    format_col_header(hrow.cells[i], h_text)

# Filas de datos
for idx, row in estudios.iterrows():
    drow = table.add_row()
    alt  = idx % 2 == 1
    cant_split_row(drow)
    for i, col in enumerate(["article", "study_design",
                               "subgroups", "techniques"]):
        format_data_cell(drow.cells[i], row.get(col, ""), alt=alt)


doc.add_paragraph()
p = doc.add_paragraph("Abreviaturas: AL = amiloidosis de cadena ligera.")
p.runs[0].italic = True
p.runs[0].font.size = Pt(9)

output = os.path.join(BASE_DIR, "tabla_características.docx")
doc.save(output)
print(f"✓ Documento generado: {output}")

✓ Documento generado: C:\Users\axdrx\OneDrive\Escritorio\TFM\Extraccion\tabla_características.docx


In [5]:
# Tabla 2: Biomarcadores

BASE_DIR = r"C:\Users\axdrx\OneDrive\Escritorio\TFM\Extraccion"
biomarks = pd.read_csv(os.path.join(BASE_DIR, "tabla_biomarcadores.csv"), encoding='utf-8-sig')

# Helpers

def set_cell_bg(cell, color_hex):
    tc   = cell._tc
    tcPr = tc.get_or_add_tcPr()
    shd  = OxmlElement('w:shd')
    shd.set(qn('w:val'),   'clear')
    shd.set(qn('w:color'), 'auto')
    shd.set(qn('w:fill'),  color_hex)
    tcPr.append(shd)

def set_cell_border_bottom(cell, dotted=False):
    tc   = cell._tc
    tcPr = tc.get_or_add_tcPr()
    tcBorders = OxmlElement('w:tcBorders')
    bottom = OxmlElement('w:bottom')
    bottom.set(qn('w:val'),   'dotted' if dotted else 'single')
    bottom.set(qn('w:sz'),    '4')
    bottom.set(qn('w:space'), '0')
    bottom.set(qn('w:color'), 'AAAAAA')
    tcBorders.append(bottom)
    tcPr.append(tcBorders)

def cant_split_row(row):
    """Evita que una fila se parta entre páginas."""
    trPr = row._tr.get_or_add_trPr()
    cs = OxmlElement('w:cantSplit')
    cs.set(qn('w:val'), '1')
    trPr.append(cs)

def format_header_row(row, text, ncols):
    """Fila gris de agrupación por artículo."""
    cell = row.cells[0]
    for i in range(1, ncols):
        cell = cell.merge(row.cells[i])
    set_cell_bg(cell, "D9D9D9")
    cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
    p = cell.paragraphs[0]
    p.paragraph_format.keep_with_next = True
    run = p.add_run(text)
    run.bold = True
    run.font.size = Pt(10.5)
    run.font.name = "Calibri"

def format_col_header(cell, text):
    """Cabecera de columna."""
    set_cell_bg(cell, "F2F2F2")
    cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
    p = cell.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.space_before = Pt(4)
    p.paragraph_format.space_after  = Pt(4)
    run = p.add_run(text)
    run.bold = True
    run.font.size = Pt(10)
    run.font.name = "Calibri"

def format_biomarker_cell(cell, name, btype, sample):
    """Primera columna: nombre en negrita + tipo (muestra) en cursiva."""
    cell.vertical_alignment = WD_ALIGN_VERTICAL.TOP
    p = cell.paragraphs[0]
    p.paragraph_format.keep_together = True
    p.paragraph_format.space_before  = Pt(4)
    p.paragraph_format.space_after   = Pt(4)
    run1 = p.add_run(str(name) if pd.notna(name) else "")
    run1.bold = True
    run1.font.size = Pt(10)
    run1.font.name = "Calibri"
    tipo_text = f"\n{btype} ({sample})" if pd.notna(btype) and pd.notna(sample) else ""
    if tipo_text:
        run2 = p.add_run(tipo_text)
        run2.italic = True
        run2.font.size = Pt(9)
        run2.font.name = "Calibri"
        run2.font.color.rgb = RGBColor(0x55, 0x55, 0x55)

def format_data_cell(cell, text):
    """Celda de datos normal."""
    cell.vertical_alignment = WD_ALIGN_VERTICAL.TOP
    p = cell.paragraphs[0]
    p.paragraph_format.keep_together = True
    p.paragraph_format.space_before  = Pt(4)
    p.paragraph_format.space_after   = Pt(4)
    run = p.add_run(str(text) if pd.notna(text) else "")
    run.font.size = Pt(10)
    run.font.name = "Calibri"

# Crear documento

doc = Document()

for section in doc.sections:
    section.page_width    = Cm(29.7)
    section.page_height   = Cm(21.0)
    section.left_margin   = Cm(1.5)
    section.right_margin  = Cm(1.5)
    section.top_margin    = Cm(2.0)
    section.bottom_margin = Cm(2.0)
    section.orientation   = 1

h = doc.add_heading("Tabla 2. Biomarcadores noveles identificados en los estudios incluidos", level=1)
h.runs[0].font.color.rgb = RGBColor(0x00, 0x00, 0x00)
h.runs[0].font.size = Pt(12)
doc.add_paragraph()

NCOLS = 4
table = doc.add_table(rows=0, cols=NCOLS)
table.style = 'Table Grid'

# Cabecera de columnas
hrow = table.add_row()
headers = ["Biomarcador\nTipo (Muestra)",
           "Propósito clínico / Fase", "Hallazgos principales", "Limitaciones"]
for i, h_text in enumerate(headers):
    format_col_header(hrow.cells[i], h_text)

# Filas

for article_id, group in biomarks.groupby("article_id", sort=False):

    # Fila gris de agrupación
    grow = table.add_row()
    format_header_row(grow, article_id, NCOLS)
    cant_split_row(grow)

    # Filas de biomarcadores
    rows_list = list(group.iterrows())
    for idx, (_, bm) in enumerate(rows_list):
        drow = table.add_row()
        cant_split_row(drow)

        format_biomarker_cell(
            drow.cells[0],
            bm.get("biomarker_name", ""),
            bm.get("biomarker_type", ""),
            bm.get("sample_type", "")
        )
        format_data_cell(drow.cells[1], bm.get("clinical_purpose", ""))
        format_data_cell(drow.cells[2], bm.get("main_findings", ""))
        format_data_cell(drow.cells[3], bm.get("limitations", ""))

        # Línea punteada entre biomarcadores del mismo artículo
        if idx < len(rows_list) - 1:
            for c in drow.cells:
                set_cell_border_bottom(c, dotted=True)

doc.add_paragraph()
p = doc.add_paragraph("Abreviaturas: AL = amiloidosis de cadena ligera; AUC = área bajo la curva ROC; HR = hazard ratio.")
p.runs[0].italic = True
p.runs[0].font.size = Pt(9)

output = os.path.join(BASE_DIR, "tabla_biomarcadores_final.docx")
doc.save(output)
print(f"✓ Documento generado: {output}")

✓ Documento generado: C:\Users\axdrx\OneDrive\Escritorio\TFM\Extraccion\tabla_biomarcadores_final.docx
